In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
from lightgbm import LGBMRegressor
warnings.filterwarnings('ignore')

# ==============================================================================
# 1. SETUP & LOAD DATA
# ==============================================================================
path_main = '/kaggle/input/agribora/agriBORA_maize_prices.csv'
path_update = '/kaggle/input/agribora/agriBORA_maize_prices_weeks_46_to_51.csv'
path_sample = '/kaggle/input/agribora/SampleSubmission (7).csv'

# Load Main
if os.path.exists(path_main):
    df_main = pd.read_csv(path_main)
else:
    df_main = pd.DataFrame()

# Load 2025 Update
if os.path.exists(path_update):
    df_update = pd.read_csv(path_update)
else:
    df_update = pd.DataFrame()

# Merge
df = pd.concat([df_main, df_update], ignore_index=True)
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['County', 'Date'])
df=df[df['Date']>'2022-01-01']
print(f"Data Loaded. Range: {df['Date'].min()} to {df['Date'].max()}")

# ==============================================================================
# 2. FEATURE ENGINEERING (1 Feature: Diff_L1)
# ==============================================================================
def get_county_data(df_in, county_name):
    c_df = df_in[df_in['County'] == county_name].sort_values('Date').set_index('Date')
    ts = c_df['WholeSale'].resample('W-MON').mean().interpolate(method='linear', limit_direction='both')
    data = pd.DataFrame({'Price': ts})

    # Create Lags
    data['Lag1'] = data['Price'].shift(1)
    data['Lag2'] = data['Price'].shift(2)

    # Feature: Momentum (P_t-1 - P_t-2)
    data['Diff_L1'] = data['Lag1'] - data['Lag2']

    # Target: Change (P_t - P_t-1)
    data['Target'] = data['Price'] - data['Lag1']

    return data.dropna()

# ==============================================================================
# 3. PREDICT (Ensemble: LGBM + RF)
# ==============================================================================
def predict_stable_diff(df, county):
    # A. Training Data
    train_data = get_county_data(df, county)

    # --- 1 Feature Only ---
    features = ['Diff_L1']

    X_train = train_data[features]
    y_train = train_data['Target']

    # B. Train Ensemble (LGBM + RF)
    models = [
        LGBMRegressor(n_estimators=50, learning_rate=0.05, verbose=-1, random_state=42),
        LGBMRegressor(n_estimators=500, learning_rate=0.05, verbose=-1, random_state=0),
        #LGBMRegressor(n_estimators=50, learning_rate=0.1, verbose=-1, random_state=34343),
         # Added RF
    ]

    for m in models:
        m.fit(X_train, y_train)

    # C. Get Anchor (Week 51)
    recent_prices = train_data['Price'].tail(2).values
    p51 = recent_prices[-1]
    p50 = recent_prices[-2]

    # Feature for Week 52 Prediction: Momentum from P50 to P51
    current_momentum = p51 - p50
    x_input = pd.DataFrame([{'Diff_L1': current_momentum}])

    # Predict Expected Change (Diff)
    preds = [m.predict(x_input)[0] for m in models]
    predicted_diff = np.mean(preds)

    # --- CALCULATE PRICES ---

    price_52 = p51 + (predicted_diff*.6)
    price_01 = p51 + (predicted_diff * 2.4)

    return price_52, price_01

# ==============================================================================
# 4. GENERATE SUBMISSION
# ==============================================================================
target_counties = ['Kiambu', 'Kirinyaga', 'Mombasa', 'Nairobi', 'Uasin-Gishu']
results_map = {}

print("\n--- Final Predictions (LGBM Ensemble) ---")
print(f"{'County':<15} | {'Week 52':<10} | {'Week 01':<10}")
print("-" * 45)

for county in target_counties:
    try:
        w52, w01 = predict_stable_diff(df, county)
        results_map[f"{county}_Week_52"] = w52
        results_map[f"{county}_Week_1"] = w01
        print(f"{county:<15} | {w52:<10.2f} | {w01:<10.2f}")
    except Exception as e:
        print(f"{county:<15} | ERROR: {e}")

# Save
if os.path.exists(path_sample):
    sub = pd.read_csv(path_sample)

    # Identify columns
    cols = sub.columns
    target_col_main = cols[1]
    target_col_mae = cols[2] if len(cols) > 2 else None

    print(f"\nWriting to: {target_col_main} AND {target_col_mae if target_col_mae else 'None'}")

    for idx, row in sub.iterrows():
        if row['ID'] in results_map:
            val = results_map[row['ID']]

            sub.at[idx, target_col_main] = val
            if target_col_mae:
                sub.at[idx, target_col_mae] = val

    sub.to_csv('sub_agro.csv', index=False)

/usr/local/lib/python3.12/dist-packages/sqlalchemy/orm/query.py:195: SyntaxWarning: "is not" with 'tuple' literal. Did you mean "!="?
  if entities is not ():


Data Loaded. Range: 2023-10-02 00:00:00 to 2025-12-15 00:00:00

--- Final Predictions (LGBM Ensemble) ---
County          | Week 52    | Week 01   
---------------------------------------------
Kiambu          | 44.73      | 45.57     
Kirinyaga       | 46.61      | 46.44     
Mombasa         | 42.53      | 43.46     
Nairobi         | 43.15      | 44.27     
Uasin-Gishu     | 41.64      | 41.54     

Writing to: Target_RMSE AND Target_MAE
